# Computation Spaces: Token CoT, Continuous Latent Reasoning, Recurrent Depth, and JEPA

[![Track](https://img.shields.io/badge/Track-PyTorch_2.x_%7C_JAX_%7C_Candle_%7C_Kernels-blue.svg)](../../README.md)
[![Parity](https://img.shields.io/badge/Parity-Transitive_%3C_1e--5-success.svg)](../../tests/test_cross_track_parity.py)

### Historical Lineage & Seminal Citations
- Wei et al. (NeurIPS 2022, Chain-of-Thought): Token chains of thought as explicit runtime compute expansion.
- Dehghani et al. (ICLR 2019, Universal Transformers) & Hao et al. (arXiv 2024, Coconut): Continuous recurrent latent reasoning without discrete token quantization bottlenecks.
- Banino et al. (NeurIPS 2021, PonderNet): Geometric-hazard adaptive halting gates for dynamic computational depth.
- LeCun (OpenReview 2022) & Assran et al. (CVPR 2023, I-JEPA): Joint-Embedding Predictive Architecture for abstract latent-space representation prediction.

### Problem Statement & Purpose
Explores the dichotomy between autoregressive discrete token deliberation and continuous latent space computation. Analyzes how continuous latent reasoning circumvents discrete sampling bottlenecks, how persistent input-injection anchoring (e_x) prevents representational collapse, how PonderNet hazards dynamically halt recurrence, and how JEPA predicts target states directly in latent space.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT use unanchored recurrent latent deliberation loops for extended reasoning steps; without persistent input residual injection (e_x @ W_anc), continuous hidden states suffer rapid representation collapse and semantic drift into low-rank attractors. Do NOT use pixel- or token-level generative reconstruction losses when abstract invariant planning is required; use JEPA self-supervised latent prediction.

### Topic Connections
🔗 Upstream: [07. Input & Output Spaces](07_input_and_output_spaces_tokens_bytes_patches_coords_pointers_and_manifolds.ipynb) | 🔗 Downstream: [09. Non-Euclidean Geometries](09_non_euclidean_geometries_hyperbolic_poincare_spherical_and_spd.ipynb)


## Mathematical Foundations & First-Principles Derivations

### 1. Continuous Latent Reasoning with Input Anchoring
Given input $\mathbf{x} \in \mathbb{R}^{B \times D_{\text{in}}}$, we compute anchor $\mathbf{e}_x = \text{GELU}(\mathbf{x} \mathbf{W}_{\text{proj}} + \mathbf{b}_{\text{proj}})$, with $\mathbf{z}^{(0)} = \mathbf{e}_x$.
For recurrence steps $k = 0, 1, \dots, K-1$, persistent input injection satisfies:
$$\mathbf{z}^{(k+1)} = \text{RMSNorm}\left( \mathbf{z}^{(k)} + \text{GELU}(\mathbf{z}^{(k)} \mathbf{W}_z + \mathbf{e}_x \mathbf{W}_{\text{anc}} + \mathbf{b}_z) \right), \quad \text{RMSNorm}(\mathbf{u}) = \frac{\mathbf{u}}{\sqrt{\frac{1}{D} \sum_{d=1}^D u_d^2 + \epsilon}}$$

### 2. Adaptive Halting Mechanics (PonderNet Hazards)
At step $k \in \{1, \dots, K\}$, hazard $\lambda_k = \sigma(\mathbf{z}^{(k)} \mathbf{w}_h + b_h)$. The halting probability is:
$$p_k = \lambda_k \prod_{j=1}^{k-1} (1 - \lambda_j), \quad \sum_{k=1}^K p_k \le 1.0$$

### 3. Joint-Embedding Predictive Architecture (JEPA)
Given context latent $\mathbf{s}_x = \mathbf{z}^{(K)}$ and target latent $\mathbf{s}_y$:
$$\hat{\mathbf{s}}_y = \text{GELU}(\mathbf{s}_x \mathbf{W}_{p1} + \mathbf{b}_{p1}) \mathbf{W}_{p2} + \mathbf{b}_{p2}, \quad \mathcal{L}_{\text{JEPA}} = \|\hat{\mathbf{s}}_y - \mathbf{s}_y\|_2^2$$

### Asymptotic Complexity
- Continuous Latent Deliberation: $\mathcal{O}(K \cdot B \cdot D^2)$ compute, $\mathcal{O}(B \cdot D)$ memory footprint (recurrent weight tying).
- PonderNet Halting: $\mathcal{O}(K \cdot B \cdot D)$ compute | JEPA Predictor: $\mathcal{O}(B \cdot D^2)$ compute.


In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd()
while not (repo_root / "core").is_dir() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import core  # noqa: E402

core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_path = repo_root / "weights" / "01_paradigms_and_dichotomies" / "08_computation_spaces_latent_reasoning_and_jepa.safetensors"
weights, ref_in, ref_out, meta = core.parity.load_reference_checkpoint(weights_path, device=device)
print(f"Device: {device} | Checkpoint: {weights_path.name} | Meta: {meta}")


In [ ]:
from jaxtyping import Float
from torch import Tensor

def rms_norm(z: Float[Tensor, "... dim"], eps: float = 1e-5) -> Float[Tensor, "... dim"]:
    return z * torch.rsqrt(torch.mean(z**2, dim=-1, keepdim=True) + eps)

class ContinuousLatentReasoner(nn.Module):
    def __init__(self, w_proj: Float[Tensor, "d_in d_model"], b_proj: Float[Tensor, "d_model"], w_z: Float[Tensor, "d_model d_model"], w_anc: Float[Tensor, "d_model d_model"], b_z: Float[Tensor, "d_model"]) -> None:
        super().__init__()
        self.w_proj, self.b_proj = nn.Parameter(w_proj.clone()), nn.Parameter(b_proj.clone())
        self.w_z, self.w_anc, self.b_z = nn.Parameter(w_z.clone()), nn.Parameter(w_anc.clone()), nn.Parameter(b_z.clone())

    def forward(self, x: Float[Tensor, "b d_in"], steps: int = 3) -> tuple[Float[Tensor, "b k_plus_1 d_model"], Float[Tensor, "b d_model"]]:
        e_x = F.gelu(x @ self.w_proj + self.b_proj)
        z, trajectories = e_x.clone(), [e_x.clone()]
        for _ in range(steps):
            z = rms_norm(z + F.gelu(z @ self.w_z + e_x @ self.w_anc + self.b_z))
            trajectories.append(z)
        return torch.stack(trajectories, dim=1), z

class PonderNetHalting(nn.Module):
    def __init__(self, w_h: Float[Tensor, "d_model 1"], b_h: Float[Tensor, "1"]) -> None:
        super().__init__()
        self.w_h, self.b_h = nn.Parameter(w_h.clone()), nn.Parameter(b_h.clone())

    def forward(self, trajectories: Float[Tensor, "b k_plus_1 d_model"]) -> Float[Tensor, "b k"]:
        batch, steps = trajectories.shape[0], trajectories.shape[1] - 1
        hazards_t = torch.cat([torch.sigmoid(trajectories[:, k + 1] @ self.w_h + self.b_h) for k in range(steps)], dim=-1)
        halting_probs, unhalted = [], torch.ones(batch, 1, device=trajectories.device, dtype=trajectories.dtype)
        for k in range(steps):
            lam = hazards_t[:, k : k + 1]
            halting_probs.append(lam * unhalted)
            unhalted = unhalted * (1.0 - lam)
        return torch.cat(halting_probs, dim=-1)

class JEPAPredictor(nn.Module):
    def __init__(self, w_p1: Float[Tensor, "d_model d_model"], b_p1: Float[Tensor, "d_model"], w_p2: Float[Tensor, "d_model d_model"], b_p2: Float[Tensor, "d_model"]) -> None:
        super().__init__()
        self.w_p1, self.b_p1 = nn.Parameter(w_p1.clone()), nn.Parameter(b_p1.clone())
        self.w_p2, self.b_p2 = nn.Parameter(w_p2.clone()), nn.Parameter(b_p2.clone())

    def forward(self, s_x: Float[Tensor, "b d_model"]) -> Float[Tensor, "b d_model"]:
        return F.gelu(s_x @ self.w_p1 + self.b_p1) @ self.w_p2 + self.b_p2

    @staticmethod
    def loss(pred: Float[Tensor, "b d_model"], target: Float[Tensor, "b d_model"]) -> Float[Tensor, "b"]:
        return torch.sum((pred - target) ** 2, dim=-1)

reasoner = ContinuousLatentReasoner(weights["w_proj"], weights["b_proj"], weights["w_z"], weights["w_anc"], weights["b_z"]).to(device)
ponder = PonderNetHalting(weights["w_h"], weights["b_h"]).to(device)
jepa = JEPAPredictor(weights["w_p1"], weights["b_p1"], weights["w_p2"], weights["b_p2"]).to(device)
print("ContinuousLatentReasoner, PonderNetHalting, and JEPAPredictor initialized successfully.")


In [ ]:
# 1. Canonical Linear Anchor Parity
w_anchor, b_anchor = weights["weight"], weights["bias"]
y_linear = ref_in @ w_anchor + b_anchor
assert float(torch.max(torch.abs(y_linear - ref_out)).item()) < 1e-5

# 2. Continuous Latent Reasoner Parity & Invariants
trajectories, final_thought = reasoner(ref_in, steps=3)
assert float(torch.max(torch.abs(trajectories - weights["latent_trajectories"])).item()) < 1e-5
assert float(torch.max(torch.abs(final_thought - weights["final_thought"])).item()) < 1e-5

# 3. PonderNet Halting Parity & Simplex Bounds
halting_probs = ponder(trajectories)
assert float(torch.max(torch.abs(halting_probs - weights["halting_probs"])).item()) < 1e-5
assert torch.all(halting_probs >= 0.0) and torch.all(halting_probs.sum(dim=-1) <= 1.0 + 1e-5)

# 4. JEPA Latent Prediction Parity & Loss
jepa_pred = jepa(final_thought)
assert float(torch.max(torch.abs(jepa_pred - weights["jepa_pred"])).item()) < 1e-5
jepa_err = jepa.loss(jepa_pred, weights["target_sy"])
assert float(torch.max(torch.abs(jepa_err - weights["jepa_error"])).item()) < 1e-5
print("All mathematical invariants (Transitive Parity < 1e-5, Halting Simplex, JEPA Consistency) verified.")


In [ ]:
# Multi-Track Micro-Benchmark Table
track_fns = {
    "Linear Anchor Projection": lambda: ref_in @ w_anchor + b_anchor,
    "Continuous Deliberation": lambda: reasoner(ref_in, steps=3),
    "PonderNet Halting Gate": lambda: ponder(trajectories),
    "JEPA Latent Predictor": lambda: jepa(final_thought),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Misuse: Unanchored Recurrent Drift vs Input-Anchored Deliberation
class UnanchoredReasoner(nn.Module):
    def __init__(self, w_z: Float[Tensor, "d d"], b_z: Float[Tensor, "d"]) -> None:
        super().__init__()
        self.w_z, self.b_z = nn.Parameter(w_z), nn.Parameter(b_z)
    def forward(self, z0: Float[Tensor, "b d"], steps: int = 20) -> Float[Tensor, "b d"]:
        z = z0
        for _ in range(steps):
            z = rms_norm(z + F.gelu(z @ self.w_z + self.b_z))
        return z

unanchored = UnanchoredReasoner(weights["w_z"], weights["b_z"]).to(device)
z_drift = unanchored(trajectories[:, 0], steps=20)
cos_drift = F.cosine_similarity(z_drift, trajectories[:, 0], dim=-1).mean().item()
cos_anchored = F.cosine_similarity(final_thought, trajectories[:, 0], dim=-1).mean().item()
print(f"[FAILURE DEMONSTRATED] Unanchored alignment collapsed to {cos_drift:.4f} vs anchored {cos_anchored:.4f}!")
assert cos_drift < cos_anchored, "Unanchored drift failed to degrade alignment"

# Correct Pattern: torch.compile(fullgraph=True) and torch.export verification
ok_exp, _ = core.export.verify_torch_export(jepa, final_thought)
ok_comp, _ = core.export.verify_torch_compile_fullgraph(jepa, final_thought)
assert ok_exp and ok_comp, "Compilation or export verification failed"
print("[CORRECT] JEPA Latent Predictor verified with torch.compile(fullgraph=True) and torch.export.")


### Mathematical & Architectural Synthesis

<svg viewBox="0 0 650 120" width="100%" height="120" xmlns="http://www.w3.org/2000/svg">
  <rect x="5" y="5" width="640" height="110" rx="8" fill="#0f172a" stroke="#334155" stroke-width="1.5"/>
  <rect x="25" y="25" width="135" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="92" y="47" fill="#38bdf8" font-size="11" font-weight="bold" text-anchor="middle">Input Anchor e_x</text>
  <rect x="175" y="25" width="135" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="242" y="47" fill="#10b981" font-size="11" font-weight="bold" text-anchor="middle">Recurrent Latents</text>
  <rect x="325" y="25" width="135" height="35" rx="6" fill="#1e293b" stroke="#f59e0b" stroke-width="1.5"/>
  <text x="392" y="47" fill="#f59e0b" font-size="11" font-weight="bold" text-anchor="middle">PonderNet Halting</text>
  <rect x="475" y="25" width="150" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="550" y="47" fill="#a855f7" font-size="11" font-weight="bold" text-anchor="middle">JEPA Latent Predictor</text>
  <rect x="25" y="70" width="600" height="32" rx="4" fill="#0f172a" stroke="#64748b" stroke-width="1.2" stroke-dasharray="4 4"/>
  <text x="325" y="90" fill="#cbd5e1" font-size="10" text-anchor="middle">Input Anchoring prevents semantic drift | Adaptive Halting optimizes runtime | JEPA learns representations</text>
</svg>

### Key Algorithmic Takeaways
1. **Continuous Superposition:** Deliberating in continuous space avoids discrete token commitment.
2. **Persistent Input Anchoring:** Injecting $\mathbf{e}_x \mathbf{W}_{\text{anc}}$ prevents hidden states from wandering into degenerate attractors.
3. **Adaptive Depth:** PonderNet hazards allocate compute adaptively per input difficulty.
4. **Abstract Representation Learning:** JEPA predicts semantic states directly in latent feature space.

---
**Curriculum Navigation:**
&larr; [Previous: 07. Input & Output Spaces](07_input_and_output_spaces_tokens_bytes_patches_coords_pointers_and_manifolds.ipynb) | [Master Index: README.md](../../README.md) | Next: 09. Non-Euclidean Geometries (Category 01 Topic 09) &rarr;
